[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C46_Graph_ML_Course/03_gat_sage/03_gat_sage.ipynb)

# 03 · GAT 与 GraphSAGE（用 numpy 从零）

目标：从零实现 **GAT 注意力层**（验证邻域内归一化、多头）与 **GraphSAGE 采样 + 聚合器**（mean/pool，验证归纳 & 采样无偏）。

路线：注意力系数(手算对拍) → 单头 GAT 层(邻域 softmax) → 多头拼接 → SAGE 聚合器(mean/pool) → 邻居采样(无偏估计) → 归纳预测新节点 → ✏️ 练习 → 📖 答案 → 🧪 SBM 归纳节点分类胶囊。

> 心智模型：**GAT = GCN 的固定边权换成可学注意力 α_ij（邻域内归一化）；SAGE = 采样固定邻居 + 可学聚合器 + 自身拼接 → 归纳、可扩展。**

## 1 · 注意力系数 α_ij：逐边打分、邻域内 softmax

GAT 给每条边算权重：`e_ij = LeakyReLU(aᵀ[Wh_i ‖ Wh_j])`，再在**每个中心节点的邻域内** softmax 归一化。
**关键**：α 在每个节点的邻居集合上归一化（每行和=1），非全图、非全边。我们手算对拍一个小例子。

In [ ]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)

def leaky_relu(x, slope=0.2):
    return np.where(x > 0, x, slope * x)

def attention_coeffs(A, Z, a_src, a_dst, slope=0.2, add_self=True):
    '''Z=Wh (已变换). a 拆成 a_src,a_dst (各 F' 维): e_ij=LeakyReLU(a_src·z_i + a_dst·z_j).
       只在邻居(+自环)上算分数，邻域内 softmax。返回稠密 α(非邻居为0)。'''
    nn = len(A)
    Aself = A + np.eye(nn) if add_self else A
    src_score = Z @ a_src                  # 每个节点作为 i 的贡献
    dst_score = Z @ a_dst                  # 每个节点作为 j 的贡献
    E = src_score[:, None] + dst_score[None, :]   # e_ij (未激活)
    E = leaky_relu(E, slope)
    E = np.where(Aself > 0, E, -1e9)       # 非邻居掩码为 -inf
    E = E - E.max(1, keepdims=True)        # 数值稳定
    expE = np.exp(E) * (Aself > 0)
    alpha = expE / expE.sum(1, keepdims=True)
    return alpha

edges = [(0,1),(0,2),(1,2),(2,3)]
n = 4; A = np.zeros((n,n))
for i,j in edges: A[i,j]=A[j,i]=1.0
rng = np.random.default_rng(0)
Z = rng.standard_normal((n, 3))
a_src = rng.standard_normal(3); a_dst = rng.standard_normal(3)
alpha = attention_coeffs(A, Z, a_src, a_dst)
print('注意力系数矩阵 α (行=中心节点):\n', alpha)
assert np.allclose(alpha.sum(1), 1.0), '每个节点 α 之和必须为 1'
assert alpha[0,3] == 0.0, '节点0与3不相连，α 应为 0'
print('✅ 邻域内归一化：每行(每节点)注意力和=1，非邻居α=0')

**手算对拍**：取节点 0（邻居含自身{0,1,2}），手动按公式算它对每个邻居的 α，与函数输出比对。

In [ ]:
i = 0; nbrs = [0,1,2]                   # 节点0 的邻居(含自环)
es = []
for j in nbrs:
    e = leaky_relu(Z[i]@a_src + Z[j]@a_dst)
    es.append(e)
es = np.array(es); es = es - es.max()
alpha_manual = np.exp(es) / np.exp(es).sum()
print('手算 α[0,{0,1,2}] =', alpha_manual)
print('函数 α[0,{0,1,2}] =', alpha[0, nbrs])
assert np.allclose(alpha_manual, alpha[0, nbrs]), '手算应与函数一致'
print('✅ 手算对拍通过：GAT 注意力公式实现正确')

## 2 · 单头 GAT 层：σ(Σ_j α_ij W h_j)

完整一层：① 线性变换 `Z=HWᵀ`；② 算 α；③ 加权聚合 `H'=σ(α @ Z)`；非线性用 ELU。
验证：输出维度正确、置换等变。

In [ ]:
def elu(x, alpha=1.0):
    return np.where(x > 0, x, alpha*(np.exp(np.minimum(x,0))-1))

def gat_layer(A, H, W, a_src, a_dst, act=elu):
    '''单头 GAT。W: (F', F). 返回 (n, F').'''
    Z = H @ W.T                            # 线性变换
    alpha = attention_coeffs(A, Z, a_src, a_dst)
    return act(alpha @ Z)                  # 加权聚合 + 非线性

F, Fp = 5, 3
rng = np.random.default_rng(1)
H = rng.standard_normal((n, F))
W = rng.standard_normal((Fp, F)) * 0.5
a_src = rng.standard_normal(Fp); a_dst = rng.standard_normal(Fp)
out = gat_layer(A, H, W, a_src, a_dst)
print('GAT 层输出形状:', out.shape)
assert out.shape == (n, Fp)
print('✅ 单头 GAT 层输出维度正确 (n, F\')')

In [ ]:
# 置换等变：重排节点 -> 输出随之重排
perm = rng.permutation(n); P = np.eye(n)[perm]
A_perm = P @ A @ P.T
out_then_perm = P @ gat_layer(A, H, W, a_src, a_dst)
perm_then_out = gat_layer(A_perm, P @ H, W, a_src, a_dst)
assert np.allclose(out_then_perm, perm_then_out, atol=1e-8), 'GAT 必须置换等变'
print('✅ GAT 层置换等变（注意力机制不依赖节点编号）')

## 3 · 多头注意力：拼接 K 个独立头

K 个头各有独立的 W,a，并行算后**拼接**（中间层）或**平均**（最后层）。拼接后维度 = K·F'。
验证：拼接维度正确、各头确实独立（不同 W 给不同输出）。

In [ ]:
def multihead_gat(A, H, heads, mode='concat', act=elu):
    '''heads: list of (W, a_src, a_dst). mode: concat(中间层) / mean(最后层).'''
    outs = [gat_layer(A, H, W, asrc, adst, act) for (W, asrc, adst) in heads]
    if mode == 'concat':
        return np.concatenate(outs, axis=1)
    else:
        return np.mean(outs, axis=0)

K = 4
heads = []
for k in range(K):
    rng_k = np.random.default_rng(10+k)
    heads.append((rng_k.standard_normal((Fp,F))*0.5,
                  rng_k.standard_normal(Fp), rng_k.standard_normal(Fp)))
out_concat = multihead_gat(A, H, heads, 'concat')
out_mean = multihead_gat(A, H, heads, 'mean')
print('拼接输出形状:', out_concat.shape, ' 平均输出形状:', out_mean.shape)
assert out_concat.shape == (n, K*Fp), '拼接后维度应为 K·F\''
assert out_mean.shape == (n, Fp), '平均后维度应为 F\''
h0 = gat_layer(A, H, *heads[0]); h1 = gat_layer(A, H, *heads[1])
assert not np.allclose(h0, h1), '不同头(不同W)应给不同输出'
print('✅ 多头：拼接维度 K·F\'、平均维度 F\'，各头独立')

## 4 · GraphSAGE 聚合器：mean 与 pool

SAGE 一层：聚合邻居 `h_N = AGG({h_u})`，再与自身**拼接**变换 `h' = σ(W·[h_v ‖ h_N])`。
实现 mean 聚合（平均）与 pool 聚合（邻居过 MLP 后逐元素 max）。

In [ ]:
def relu(x): return np.maximum(x, 0.0)

def neighbors(A): return [np.nonzero(A[i])[0] for i in range(len(A))]

def sage_mean_agg(A, H):
    '''mean 聚合邻居(不含自己)。'''
    nbrs = neighbors(A)
    out = np.zeros_like(H)
    for v in range(len(H)):
        if len(nbrs[v]) > 0:
            out[v] = H[nbrs[v]].mean(0)
    return out

def sage_pool_agg(A, H, W_pool, b_pool):
    '''pool 聚合：邻居过 MLP(relu) 后逐元素 max。'''
    nbrs = neighbors(A)
    out = np.zeros((len(H), W_pool.shape[0]))
    for v in range(len(H)):
        if len(nbrs[v]) > 0:
            transformed = relu(H[nbrs[v]] @ W_pool.T + b_pool)
            out[v] = transformed.max(0)
    return out

def sage_layer(A, H, W, agg='mean', W_pool=None, b_pool=None, act=relu):
    '''W 作用在 [h_v ‖ h_N] 上。'''
    h_nbr = sage_mean_agg(A, H) if agg=='mean' else sage_pool_agg(A, H, W_pool, b_pool)
    concat = np.concatenate([H, h_nbr], axis=1)   # 自身 ‖ 邻域
    out = act(concat @ W.T)
    return out / (np.linalg.norm(out, axis=1, keepdims=True) + 1e-12)  # L2 归一化

edges = [(0,1),(0,2),(1,2),(2,3),(3,4)]
n = 5; A = np.zeros((n,n))
for i,j in edges: A[i,j]=A[j,i]=1.0
rng = np.random.default_rng(2)
H = rng.standard_normal((n, 4))
W = rng.standard_normal((6, 8)) * 0.3      # 输入 = [h_v(4)‖h_N(4)] = 8 维
out = sage_layer(A, H, W, agg='mean')
print('SAGE-mean 输出形状:', out.shape)
assert out.shape == (n, 6)
assert np.allclose(np.linalg.norm(out, axis=1), 1.0), 'SAGE 输出应 L2 归一化'
hn = sage_mean_agg(A, H)
assert np.allclose(hn[0], (H[1]+H[2])/2)    # 节点0邻居{1,2}
print('✅ SAGE-mean 层：自身‖邻域拼接、L2 归一化，mean 聚合正确')

In [ ]:
# pool 聚合器手算对拍
W_pool = rng.standard_normal((4, 4))*0.5; b_pool = np.zeros(4)
hn_pool = sage_pool_agg(A, H, W_pool, b_pool)
manual = relu(H[[1,2]] @ W_pool.T + b_pool).max(0)   # 节点0邻居{1,2}
assert np.allclose(hn_pool[0], manual), 'pool 聚合应为邻居过MLP后max'
print('✅ SAGE-pool 聚合器：邻居过 MLP 后逐元素 max，手算对拍通过')

## 5 · 邻居采样：mean 聚合下是无偏估计

大图上每节点只采样固定数目邻居。验证（mean 聚合下）：**多次采样的均值 → 全邻域均值**（无偏）。
这是 SAGE 能 mini-batch 训练大图的根基。

In [ ]:
def sample_neighbors(A, num_samples, seed=0):
    '''每个节点随机采样 num_samples 个邻居(有放回)。'''
    rng = np.random.default_rng(seed)
    nbrs = neighbors(A)
    sampled = []
    for v in range(len(A)):
        if len(nbrs[v]) == 0:
            sampled.append(np.array([], dtype=int))
        else:
            sampled.append(rng.choice(nbrs[v], size=num_samples, replace=True))
    return sampled

# 造一个高度数节点(节点0连所有)
rng = np.random.default_rng(3)
nb = 30
Abig = np.zeros((nb+1, nb+1))
for j in range(1, nb+1): Abig[0,j]=Abig[j,0]=1.0
Hbig = rng.standard_normal((nb+1, 4))
full_mean = Hbig[1:].mean(0)               # 节点0 全邻域均值
ests = []
for s in range(2000):
    samp = sample_neighbors(Abig, num_samples=5, seed=s)
    ests.append(Hbig[samp[0]].mean(0))
mc_mean = np.mean(ests, axis=0)
print('全邻域均值      :', full_mean)
print('2000 次采样均值 :', mc_mean)
assert np.allclose(mc_mean, full_mean, atol=0.05), '采样均值应收敛到全邻域均值(无偏)'
print('✅ 邻居采样(mean)是无偏估计：多次采样的均值收敛到全邻域均值')

## 6 · 归纳：用学好的聚合函数预测「新节点」

SAGE 的精髓：学的是**聚合函数**而非具体节点嵌入，所以能给**训练时没见过的新节点**生成表示。
用固定的聚合权重给一个新加入的节点算表示——这正是直推 GCN 做不到的。

In [ ]:
edges = [(0,1),(0,2),(1,2),(2,3),(3,4)]
n = 5; A = np.zeros((n,n))
for i,j in edges: A[i,j]=A[j,i]=1.0
rng = np.random.default_rng(2)
H = rng.standard_normal((n, 4))
W = rng.standard_normal((6, 8))*0.3

def sage_embed_one(new_feat, new_nbr_feats, W, act=relu):
    '''用学好的 W 给一个新节点算嵌入：聚合它的邻居 + 自身拼接。'''
    h_nbr = new_nbr_feats.mean(0) if len(new_nbr_feats) else np.zeros_like(new_feat)
    concat = np.concatenate([new_feat, h_nbr])
    out = act(concat @ W.T)
    return out / (np.linalg.norm(out) + 1e-12)

# 新节点 5：特征随机，连到老节点 2 和 4
new_feat = rng.standard_normal(4)
new_nbr_feats = H[[2, 4]]
z_new = sage_embed_one(new_feat, new_nbr_feats, W)
print('新节点嵌入(无需重训):', z_new)
assert z_new.shape == (6,) and abs(np.linalg.norm(z_new)-1) < 1e-6
print('✅ 归纳成功：用学好的聚合函数，直接给训练时未见的新节点生成表示')

---
## ✏️ 练习 1：注意力熵（注意力有多「集中」）

对每个节点，它的注意力分布 `α[i, :]` 的熵 `H_i = -Σ_j α_ij log α_ij`（只对邻居）衡量它「平均地听」还是「集中听某个邻居」。
实现 `attention_entropy(alpha)` 返回每节点的熵向量。验证：均匀注意力熵最大。

In [ ]:
def attention_entropy(alpha):
    # TODO: 对每行(每节点)算 -Σ α log α（α=0 的项贡献0，用 where 避免 log0）
    #       返回长度 n 的熵向量
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
alpha_unif = np.array([[0.5,0.5,0,0],[1/3,1/3,1/3,0],[0,0,0.5,0.5],[0,0,1.0,0]])
ent = attention_entropy(alpha_unif)
assert np.isclose(ent[0], np.log(2)), '2个邻居均匀 -> log2'
assert np.isclose(ent[1], np.log(3)), '3个邻居均匀 -> log3'
assert np.isclose(ent[3], 0.0), '只听1个邻居 -> 熵0'
print('每节点注意力熵:', ent.round(4))
print('✅ 练习 1 通过：均匀注意力熵最大，集中注意力熵小')

## ✏️ 练习 2：邻域内 softmax（GAT 归一化核心）

实现 `neighborhood_softmax(scores, A)`：给定每条边的原始分数矩阵 `scores` 和邻接 `A`（已含需要的自环），
在**每个节点的邻居范围内**做 softmax，非邻居置 0。返回归一化的 α。

In [ ]:
def neighborhood_softmax(scores, A):
    # TODO: 把非邻居(A==0)的分数设为 -1e9；逐行减max稳定；exp 后乘(A>0)掩码；逐行归一化
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
Atest = np.array([[1.,1,0],[1,1,1.],[0,1,1.]])   # 含自环
scores = np.array([[2.,1,0],[1,3,1.],[0,1,2.]])
al = neighborhood_softmax(scores, Atest)
assert np.allclose(al.sum(1), 1.0), '每行和为1'
assert al[0,2] == 0 and al[2,0] == 0, '非邻居α=0'
exp = np.exp(np.array([2.,1.])-2); ref = exp/exp.sum()    # 节点0邻居{0,1}
assert np.allclose(al[0,[0,1]], ref)
print('✅ 练习 2 通过：邻域内 softmax 正确（GAT 注意力归一化的核心）')

## ✏️ 练习 3：max 聚合器

实现 `max_aggregate(A, H)`：每个节点取邻居特征的**逐元素最大值**（不含自己，孤立点输出0）。
max 聚合能捕捉「邻域里是否存在某特征」，比 mean 更有判别力（GIN 的前奏）。

In [ ]:
def max_aggregate(A, H):
    # TODO: 对每个节点，取其邻居特征的逐元素 max；无邻居则全 0
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
Hmax = max_aggregate(A, H)
assert np.allclose(Hmax[0], np.maximum(H[1], H[2])), 'max 聚合应逐元素取最大'  # 节点0邻居{1,2}
assert Hmax.shape == H.shape
assert not np.allclose(Hmax, sage_mean_agg(A, H)), 'max 与 mean 应不同'
print('✅ 练习 3 通过：max 聚合器逐元素取邻居最大值')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def attention_entropy(alpha):
    safe = np.where(alpha > 0, alpha, 1.0)     # log(1)=0，不影响
    return -(alpha * np.log(safe)).sum(1)

In [ ]:
# 练习 2 参考答案
def neighborhood_softmax(scores, A):
    E = np.where(A > 0, scores, -1e9)
    E = E - E.max(1, keepdims=True)
    expE = np.exp(E) * (A > 0)
    return expE / expE.sum(1, keepdims=True)

In [ ]:
# 练习 3 参考答案
def max_aggregate(A, H):
    nbrs = neighbors(A)
    out = np.zeros_like(H)
    for v in range(len(H)):
        if len(nbrs[v]) > 0:
            out[v] = H[nbrs[v]].max(0)
    return out

---
## 🧪 真实数据胶囊：归纳节点分类（在新节点上测试 SAGE 思想）

归纳学习的真正考验：在**一部分节点**上训练，在**训练时完全隐藏的节点**上测试（不重训）。
用 SBM 造一张有社区的图，分成「训练节点」和「测试节点」，验证 SAGE 式聚合学到的是**可泛化的函数**。

In [ ]:
def make_sbm(sizes, p_in, p_out, seed=0):
    rng = np.random.default_rng(seed)
    nn = sum(sizes)
    labels = np.concatenate([np.full(s,k) for k,s in enumerate(sizes)])
    A = np.zeros((nn,nn))
    for i in range(nn):
        for j in range(i+1,nn):
            if rng.random() < (p_in if labels[i]==labels[j] else p_out):
                A[i,j]=A[j,i]=1.0
    return A, labels

def softmax(z):
    z = z - z.max(1, keepdims=True); e=np.exp(z); return e/e.sum(1,keepdims=True)

A_full, y_full = make_sbm([30,30], 0.35, 0.04, seed=5)
nn = len(A_full); C = 2
rng = np.random.default_rng(6)
protos = rng.standard_normal((C, 6))
X_full = np.array([protos[y_full[i]] for i in range(nn)]) + 0.8*rng.standard_normal((nn,6))
perm = rng.permutation(nn)
test_idx = perm[:int(0.2*nn)]; train_idx = perm[int(0.2*nn):]
print(f'图 {nn} 节点：训练 {len(train_idx)}，归纳测试 {len(test_idx)}')
print('✅ 归纳划分就绪（测试节点训练时不参与损失）')

**🧪 胶囊练习**：实现一个 SAGE-mean 编码（自身‖邻域）+ 线性分类器，**只在训练节点上算损失**，在隐藏的测试节点上评估。
提示：聚合用全图邻接（测试节点的邻居可见，但其标签不可见——这才是归纳）。补全聚合那一行。

In [ ]:
def train_inductive_sage(A, X, y, train_idx, test_idx, epochs=300, lr=0.2, seed=0):
    rng = np.random.default_rng(seed)
    nn = len(A); C = len(set(y.tolist()))
    h_nbr = None    # TODO: sage_mean_agg(A, X) —— 聚合邻居(结构信息)
    feat = np.concatenate([X, h_nbr], axis=1)     # [自身 ‖ 邻域]
    W = rng.standard_normal((feat.shape[1], C))*0.1; b=np.zeros(C)
    tr = np.zeros(nn,bool); tr[train_idx]=True
    for _ in range(epochs):
        Z = feat @ W + b; P = softmax(Z)
        dZ = P.copy(); dZ[tr, y[tr]] -= 1; dZ[~tr]=0; dZ/=tr.sum()
        W -= lr*(feat.T @ dZ); b -= lr*dZ.sum(0)
    pred = (feat @ W + b).argmax(1)
    return (pred[test_idx]==y[test_idx]).mean()

raise NotImplementedError  # 删除并补全 h_nbr

In [ ]:
# 自测
acc = train_inductive_sage(A_full, X_full, y_full, train_idx, test_idx)
print(f'归纳测试准确率(隐藏节点) = {acc:.2%}')
assert acc >= 0.8, 'SAGE 聚合应能泛化到训练时未参与损失的节点'
print('✅ 胶囊练习通过：学到的聚合函数泛化到了新节点（归纳学习）')

In [ ]:
# 📖 胶囊参考答案
def train_inductive_sage(A, X, y, train_idx, test_idx, epochs=300, lr=0.2, seed=0):
    rng = np.random.default_rng(seed)
    nn = len(A); C = len(set(y.tolist()))
    h_nbr = sage_mean_agg(A, X)                   # 聚合邻居结构信息
    feat = np.concatenate([X, h_nbr], axis=1)
    W = rng.standard_normal((feat.shape[1], C))*0.1; b=np.zeros(C)
    tr = np.zeros(nn,bool); tr[train_idx]=True
    for _ in range(epochs):
        Z = feat @ W + b; P = softmax(Z)
        dZ = P.copy(); dZ[tr, y[tr]] -= 1; dZ[~tr]=0; dZ/=tr.sum()
        W -= lr*(feat.T @ dZ); b -= lr*dZ.sum(0)
    pred = (feat @ W + b).argmax(1)
    return (pred[test_idx]==y[test_idx]).mean()

acc = train_inductive_sage(A_full, X_full, y_full, train_idx, test_idx)
print(f'归纳测试准确率 = {acc:.2%}')
assert acc >= 0.8
print('✅ 聚合「自身‖邻域」学到的规则，泛化到了训练时未见标签的节点')

### 小结
- **GAT = GCN 固定边权 → 可学注意力 α_ij**：逐边打分 + **邻域内 softmax**(每行和=1) + 多头(拼接/平均)。
- **GraphSAGE = 采样固定邻居 + 可学聚合器(mean/pool/LSTM) + 自身‖邻域拼接 + L2 归一化**。
- **邻居采样**(mean 下无偏)把感受野从「度^L」压成固定值 → mini-batch 大图训练。
- **归纳**：学聚合函数而非节点嵌入 → 直接给新节点生成表示(直推 GCN 做不到)。

下一站：**模块 04 · 图 Transformer** —— 用全局注意力 + 拉普拉斯位置编码，绕开本模块所有模型共有的「局部性」与过挤压。